# Exhibit 21 Parent–Subsidiary Pipeline: run in Google Colab

Run the cells top to bottom (**Runtime → Run all** works too).

1. Upload `ex21_pipeline.zip` (the whole project) and unzip it
2. Install packages and run the tests
3. Build the report from the bundled Apple and Microsoft filings
4. Look at the results
5. *(Optional)* download more companies from SEC EDGAR
6. Download the outputs

## 1. Upload and unzip the project
When the **Choose Files** button appears, pick `ex21_pipeline.zip`.

If the project is on GitHub, skip the upload and use `!git clone <repo-url>` instead, then continue at step 2.

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()                      # choose ex21_pipeline.zip
zip_name = next(n for n in uploaded if n.endswith(".zip"))
zipfile.ZipFile(zip_name).extractall("/content")
os.chdir("/content/ex21_pipeline")
print("Working folder:", os.getcwd())
!ls

## 2. Install packages and run the format tests

In [ ]:
!pip install -q -r requirements.txt
!python tests/test_parse.py

## 3. Build the report
Reads every Exhibit 21 in `data/raw/` and writes `output/parent_subsidiary_report.xlsx` and `output/graph_edges.csv`.

In [ ]:
!python run_pipeline.py

## 4. Look at the results
These are the same sheets as in the Excel file.

In [ ]:
import pandas as pd
report = "output/parent_subsidiary_report.xlsx"
sheets = pd.read_excel(report, sheet_name=None)
print("Sheets:", list(sheets))

print("\n=== Parent → Subsidiaries (latest year per company) ===")
display(sheets["Parent → Subsidiaries"])

In [ ]:
print("=== Year-over-year changes ===")
display(sheets["Changes"])

print("=== QA: did every file parse? ===")
display(sheets["QA"])

In [ ]:
edges = pd.read_csv("output/graph_edges.csv")
print(len(edges), "edges")
display(edges.head(10))

## 5. *(Optional)* Add more companies from SEC EDGAR
Change the tickers, and put **your own** name and email in `SEC_USER_AGENT` (SEC requires a contact). Then re-run step 3 and step 4.

The **Summary** sheet's counts are Excel formulas, so they show blank in pandas until the file is opened in Excel or Google Sheets. The other sheets show normally.

In [ ]:
import os
os.environ["SEC_USER_AGENT"] = "Your Name your.email@school.edu"   # <-- edit this
TICKERS = "NVDA JPM"                                                 # <-- edit this
!python fetch_edgar.py {TICKERS} --years 3
!cat data/fetch_log.csv

In [ ]:
!python run_pipeline.py

## 6. Download the outputs

In [ ]:
files.download("output/parent_subsidiary_report.xlsx")
files.download("output/graph_edges.csv")